# Explore 2026 pitch dataset

Load MLB / MiLB Parquet pulls and sketch quick pitcher evaluations.

In [ ]:
from pathlib import Path

import pandas as pd

from pitch_dataset import DEFAULT_SEASON
from pitch_dataset.storage import read_pitches

print("default season:", DEFAULT_SEASON)
data = Path("../data")
mlb_path = data / f"pitches_mlb_{DEFAULT_SEASON}.parquet"
minors_path = data / f"pitches_minors_{DEFAULT_SEASON}.parquet"

frames = []
for path in (mlb_path, minors_path):
    if path.exists():
        frames.append(read_pitches(path))
        print(path.name, len(frames[-1]))
    else:
        print("missing:", path, "— run: uv run pitch-dataset sample")

pitches = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
pitches.head()

In [ ]:
if not pitches.empty:
    cols = [c for c in ["player_name", "pitch_type", "release_speed", "release_spin_rate", "league"] if c in pitches.columns]
    display = pitches[cols].dropna(subset=[c for c in cols if c != "player_name"]).head(20)
    display
else:
    print("No pitch rows loaded yet.")

In [ ]:
if not pitches.empty and {"player_name", "release_speed", "pitch_type"}.issubset(pitches.columns):
    summary = (
        pitches.groupby(["league", "player_name", "pitch_type"], dropna=False)
        .agg(
            pitches=("release_speed", "size"),
            avg_velo=("release_speed", "mean"),
            avg_spin=("release_spin_rate", "mean") if "release_spin_rate" in pitches.columns else ("release_speed", "size"),
        )
        .reset_index()
        .sort_values("pitches", ascending=False)
    )
    summary.head(25)
else:
    print("Need player_name / release_speed / pitch_type columns.")